In [41]:
# Here we're going to make some data merge of our datasets 

import numpy as np
import pandas as pd
import os 

from trading_data_classes import GetDataTradingView, DataWorks, Strategy
dw = DataWorks()
s = Strategy()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [27]:
# Retrieve the last 500 rows, and then (just in case) sort by timestamp and drop duplicates
# To avoid potential mistakes in case of excessive data writing 

df_TONUSDT = pd.read_csv('data/history/tonusdt.csv').tail(500)\
    .sort_values('timestamp_utc')\
    .drop_duplicates('timestamp_utc')

df_BTCUSDT = pd.read_csv('data/history/btcusdt.csv').tail(500).\
    sort_values('timestamp_utc').\
    drop_duplicates('timestamp_utc')

df_MAG7 = pd.read_csv('data/history/mag7.csv').tail(500).\
    sort_values('timestamp_utc').\
    drop_duplicates('timestamp_utc')

In [28]:
print(len(df_TONUSDT))
print(df_TONUSDT.timestamp_utc.min())
print(df_TONUSDT.timestamp_utc.max())

print(len(df_BTCUSDT))
print(df_BTCUSDT.timestamp_utc.min())
print(df_BTCUSDT.timestamp_utc.max())

print(len(df_MAG7))
print(df_MAG7.timestamp_utc.min())
print(df_MAG7.timestamp_utc.max())

500
2025-09-19 00:15:00+00:00
2025-09-20 17:50:00+00:00
500
2025-09-19 00:15:00+00:00
2025-09-20 17:50:00+00:00
500
2025-09-12 10:05:00+00:00
2025-09-19 15:25:00+00:00


In [29]:
# We can compare BTCUSDT to TONUSDT simply by the timestamps of a price
df_BTC_TON = df_BTCUSDT.merge(df_TONUSDT, on=['timestamp_utc'], suffixes=['_btc', '_ton'])
print(len(df_BTC_TON)) # Intersection of first 2

# MAG7 has trading window (does not trades on weekends), so intersection is not complete
df_BTC_TON_MAG = df_BTC_TON.merge(df_MAG7, on=['timestamp_utc'], suffixes=['_', '_mag7'])
print(len(df_BTC_TON_MAG)) # Intersection of all 3

500
77


In [30]:
df_BTC_TON.tail(5).T

,495,496,497,498,499
instrument_btc,BINANCE:BTCUSDT,BINANCE:BTCUSDT,BINANCE:BTCUSDT,BINANCE:BTCUSDT,BINANCE:BTCUSDT
timestamp_utc,2025-09-20 17:30:00+00:00,2025-09-20 17:35:00+00:00,2025-09-20 17:40:00+00:00,2025-09-20 17:45:00+00:00,2025-09-20 17:50:00+00:00
open_price_btc,115854.77,115886.84,115889.69,115877.35,115834.02
high_price_btc,115886.85,115891.97,115889.7,115877.36,115834.02
low_price_btc,115854.77,115882.87,115876.95,115834.01,115817.28
close_price_btc,115886.84,115889.69,115877.36,115834.02,115833.99
record_timestamp_utc_btc,2025-09-20 17:53:42,2025-09-20 17:53:42,2025-09-20 17:53:42,2025-09-20 17:53:42,2025-09-20 17:53:42
instrument_ton,BINANCE:TONUSDT,BINANCE:TONUSDT,BINANCE:TONUSDT,BINANCE:TONUSDT,BINANCE:TONUSDT
open_price_ton,3.095,3.093,3.095,3.094,3.092
high_price_ton,3.096,3.096,3.096,3.094,3.093


In [31]:
# # Some basic trading signals calculation functions

# def calculate_rsi(prices, period):
#     delta = prices.diff()
#     gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
#     loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
#     rs = gain / loss
#     rsi = 100 - (100 / (1 + rs))
#     return rsi

# def calculate_bollinger_bands(prices, period, std_dev):
#     sma = prices.rolling(window=period).mean()
#     rolling_std = prices.rolling(window=period).std()
#     upper_band = sma + (rolling_std * std_dev)
#     lower_band = sma - (rolling_std * std_dev)
#     return sma, upper_band, lower_band
    
# def apply_values_for_double_strat(data, close_price_field, instrument):   
#     RSI_period = 3
#     Bollinger_period = 165
#     Bollinger_std_dev = 2
#     RSI_overbought = 50
#     RSI_oversold = 49
    
#     data['RSI_dd_strat'] = calculate_rsi(data[close_price_field], RSI_period)
#     data['BB_basis'], data['BB_upper'], data['BB_lower']= calculate_bollinger_bands(
#         data[close_price_field], 
#         Bollinger_period, 
#         Bollinger_std_dev)

#     buy_signal_field_name = 'buy_signal_' + instrument
#     sell_signal_field_name = 'sell_signal_' + instrument

#     data[buy_signal_field_name] = ((data['RSI_dd_strat'] > RSI_oversold) 
#                                         & (data[close_price_field] < data['BB_lower'])).astype('int')
#     data[sell_signal_field_name] = ((data['RSI_dd_strat'] < RSI_overbought) 
#                                         & (data[close_price_field] > data['BB_upper'])).astype('int')
#     return data

In [32]:
# Since 2025-09-27 we use s.apply_values_for_double_strat() (function from our class) for test purposes 
df_BTC_TON = s.apply_values_for_double_strat(df_BTC_TON, 'close_price_btc', 'btc')
df_BTC_TON = s.apply_values_for_double_strat(df_BTC_TON, 'close_price_ton', 'ton')

# df_BTC_TON_MAG = apply_values_for_double_strat(df_BTC_TON, 'close_price', 'mag7') 
# # Excluded cause has no apropriate prices in interval 

In [33]:
df_BTC_TON.sum() # Do we have any signals at this interval? 



instrument_btc              BINANCE:BTCUSDTBINANCE:BTCUSDTBINANCE:BTCUSDTB...
timestamp_utc               2025-09-19 00:15:00+00:002025-09-19 00:20:00+0...
open_price_btc                                                    58038777.61
high_price_btc                                                    58053890.05
low_price_btc                                                     58022899.68
close_price_btc                                                   58037542.13
record_timestamp_utc_btc    2025-09-20 17:53:422025-09-20 17:53:422025-09-...
instrument_ton              BINANCE:TONUSDTBINANCE:TONUSDTBINANCE:TONUSDTB...
open_price_ton                                                       1561.122
high_price_ton                                                       1562.246
low_price_ton                                                        1559.983
close_price_ton                                                      1561.069
record_timestamp_utc_ton    2025-09-20 17:53:422025-09-20 17:53:

In [34]:
# Yes, we have some signals in a short test sample: 
# buy_signal_btc                                                              4
# sell_signal_btc                                                             1
# buy_signal_ton                                                              3
# sell_signal_ton                                                             1

In [35]:
# Let's take a look at all signals we have 

df_BTC_TON[(df_BTC_TON['buy_signal_btc'] > 0) | (df_BTC_TON['buy_signal_ton'] > 0) |
            (df_BTC_TON['sell_signal_btc'] > 0) | (df_BTC_TON['sell_signal_ton'] > 0)]

,instrument_btc,timestamp_utc,open_price_btc,high_price_btc,low_price_btc,close_price_btc,record_timestamp_utc_btc,instrument_ton,open_price_ton,high_price_ton,low_price_ton,close_price_ton,record_timestamp_utc_ton,RSI_dd_strat,BB_basis,BB_upper,BB_lower,buy_signal_btc,sell_signal_btc,buy_signal_ton,sell_signal_ton
181,BINANCE:BTCUSDT,2025-09-19 15:20:00+00:00,115560.74,115734.37,115542.00,115734.36,2025-09-20 17:53:42,BINANCE:TONUSDT,3.108,3.115,3.106,3.115,2025-09-20 17:53:42,75.000000,3.151921,3.199261,3.104582,1,0,0,0
182,BINANCE:BTCUSDT,2025-09-19 15:25:00+00:00,115734.36,115800.00,115644.00,115800.00,2025-09-20 17:53:42,BINANCE:TONUSDT,3.116,3.117,3.110,3.117,2025-09-20 17:53:42,100.000000,3.151491,3.198801,3.104181,1,0,0,0
183,BINANCE:BTCUSDT,2025-09-19 15:30:00+00:00,115800.00,115800.00,115640.00,115673.40,2025-09-20 17:53:42,BINANCE:TONUSDT,3.116,3.116,3.106,3.107,2025-09-20 17:53:42,50.000000,3.151024,3.198563,3.103486,1,0,0,0
184,BINANCE:BTCUSDT,2025-09-19 15:35:00+00:00,115673.41,115838.85,115646.51,115813.03,2025-09-20 17:53:42,BINANCE:TONUSDT,3.107,3.117,3.106,3.116,2025-09-20 17:53:42,52.380952,3.150576,3.198031,3.103121,1,0,0,0
313,BINANCE:BTCUSDT,2025-09-20 02:20:00+00:00,115444.01,115524.67,115444.00,115514.56,2025-09-20 17:53:42,BINANCE:TONUSDT,3.088,3.094,3.085,3.093,2025-09-20 17:53:42,66.666667,3.117127,3.136931,3.097323,0,0,1,0
314,BINANCE:BTCUSDT,2025-09-20 02:25:00+00:00,115514.57,115533.83,115514.56,115533.82,2025-09-20 17:53:42,BINANCE:TONUSDT,3.094,3.095,3.093,3.094,2025-09-20 17:53:42,70.000000,3.116915,3.136956,3.096874,0,0,1,0
315,BINANCE:BTCUSDT,2025-09-20 02:30:00+00:00,115533.82,115533.83,115496.71,115503.06,2025-09-20 17:53:42,BINANCE:TONUSDT,3.094,3.095,3.087,3.089,2025-09-20 17:53:42,58.333333,3.116679,3.137110,3.096248,0,0,1,0
381,BINANCE:BTCUSDT,2025-09-20 08:00:00+00:00,115968.56,115968.57,115923.12,115923.12,2025-09-20 17:53:42,BINANCE:TONUSDT,3.107,3.107,3.101,3.101,2025-09-20 17:53:42,37.500000,3.106624,3.125551,3.087698,0,1,0,0
471,BINANCE:BTCUSDT,2025-09-20 15:30:00+00:00,116110.01,116119.30,116044.69,116044.70,2025-09-20 17:53:42,BINANCE:TONUSDT,3.109,3.110,3.106,3.108,2025-09-20 17:53:42,0.000000,3.095636,3.106664,3.084608,0,0,0,1


In [ ]:
df_BTC_TON.tail(1)

,instrument_btc,timestamp_utc,open_price_btc,high_price_btc,low_price_btc,close_price_btc,record_timestamp_utc_btc,instrument_ton,open_price_ton,high_price_ton,low_price_ton,close_price_ton,record_timestamp_utc_ton,RSI_dd_strat,BB_basis,BB_upper,BB_lower,buy_signal_btc,sell_signal_btc,buy_signal_ton,sell_signal_ton
499,BINANCE:BTCUSDT,2025-09-20 17:50:00+00:00,115834.02,115834.02,115817.28,115833.99,2025-09-20 17:53:42,BINANCE:TONUSDT,3.092,3.093,3.089,3.093,2025-09-20 17:53:42,33.333333,3.09657,3.108568,3.084571,0,0,0,0


In [55]:
df_BTC_TON.iloc[471:472]

,instrument_btc,timestamp_utc,open_price_btc,high_price_btc,low_price_btc,close_price_btc,record_timestamp_utc_btc,instrument_ton,open_price_ton,high_price_ton,low_price_ton,close_price_ton,record_timestamp_utc_ton,RSI_dd_strat,BB_basis,BB_upper,BB_lower,buy_signal_btc,sell_signal_btc,buy_signal_ton,sell_signal_ton
471,BINANCE:BTCUSDT,2025-09-20 15:30:00+00:00,116110.01,116119.3,116044.69,116044.7,2025-09-20 17:53:42,BINANCE:TONUSDT,3.109,3.11,3.106,3.108,2025-09-20 17:53:42,0.0,3.095636,3.106664,3.084608,0,0,0,1


In [ ]:
# Check if the last row in df_BTC_TON satisfies the signal condition and write to signals.csv if true

last_row = df_BTC_TON.tail(1)

# For check of recording
# last_row = df_BTC_TON.iloc[471:472]

signal_condition = ((last_row['buy_signal_btc'] > 0) | (last_row['buy_signal_ton'] > 0) |
                    (last_row['sell_signal_btc'] > 0) | (last_row['sell_signal_ton'] > 0)).any()
if signal_condition:
    last_row.to_csv('data/signals.csv', mode='a', header=not os.path.exists('data/signals.csv'), index=False)
    print('Signal row written to signals.csv')
else:
    print('No trading signals this time >_< ')

No signal in the last row
